# Bakta Proteins CTS Demo

End-to-end test of `cdm_bakta_proteins:0.1.0`. Variant of `cdm_bakta` that takes a protein FASTA with pre-existing locus tags and annotates without re-predicting genes, preserving the caller's identifiers.

- **Image:** `ghcr.io/kbaseincubator/cdm_bakta_proteins:0.1.0@sha256:ba41718caf200d897c01e53b2a0d58f953b30a8e15d1435b516f912f0142d89e`
- **Refdata UUID:** `30f8ba11-a456-408c-a9f9-7d232ba3ed8e` (same bundle as `cdm_bakta`)
- **Cluster:** `kbase`
- **Output:** `cts/io/jplfaria/output/bakta_proteins/test/v1`

**Input:** prodigal-called protein FASTA from checkm2's earlier run (`cts/io/gavin/test_output/kb_demo/.../*.faa`). Headers look like `>AE017199.1_1 # 1 # 879 # ...`, where `AE017199.1_1` is the prodigal-style locus tag we expect to see preserved in bakta_proteins's output.

**What success looks like:** the output `.tsv` from bakta_proteins lists each locus exactly as it appears in the input FASTA (`AE017199.1_1`, `AE017199.1_2`, ...). If bakta_proteins were generating its own tags (like `cdm_bakta` does), we'd see `AHLLAC_0001`-style identifiers instead, which would mean Mode B isn't actually preserving the input IDs.

## 1. Setup

In [1]:
import io, json, time
import pandas as pd

tscli  = get_task_service_client()
mincli = get_minio_client()

IMAGE = "ghcr.io/kbaseincubator/cdm_bakta_proteins:0.1.0@sha256:ba41718caf200d897c01e53b2a0d58f953b30a8e15d1435b516f912f0142d89e"
OUTPUT_DIR = "cts/io/jplfaria/output/bakta_proteins/test/v1"

print(tscli.whoami())

{'user': 'jplfaria', 'roles': [], 'allowed_paths': [{'path': 'cts/io/', 'perm': 'write'}]}


## 2. List input protein FASTA files

Use the 4 prodigal-called protein FASTA files from checkm2's run (these carry prodigal-style locus tags like `AE017199.1_1`). Each `.faa` is one input genome's proteins.

In [1]:
input_files = []
for o in mincli.list_objects("cts", prefix="io/gavin/test_output/kb_demo", recursive=True):
    if o.object_name.endswith(".faa"):
        input_files.append(f"cts/{o.object_name}")

print(f"{len(input_files)} protein FASTA file(s):")
for f in input_files:
    print(f"  {f}")

# Spot-check the input locus tags so we know what to look for in the output
sample_key = input_files[0].split("/", 1)[1]
raw = mincli.get_object("cts", sample_key).read().decode("utf-8", errors="replace")
sample_tags = [l[1:].split()[0] for l in raw.splitlines() if l.startswith(">")][:5]
print(f"\nfirst 5 locus tags in {sample_key}:")
for t in sample_tags:
    print(f"  {t}")

4 protein FASTA file(s):
  cts/io/gavin/test_output/kb_demo/0/protein_files/GCA_000008085.1_ASM808v1_genomic.faa
  cts/io/gavin/test_output/kb_demo/0/protein_files/GCA_000010565.1_ASM1056v1_genomic.faa
  cts/io/gavin/test_output/kb_demo/1/protein_files/GCA_000145985.1_ASM14598v1_genomic.faa
  cts/io/gavin/test_output/kb_demo/1/protein_files/GCA_000147015.1_ASM14701v1_genomic.faa

first 5 locus tags in io/gavin/test_output/kb_demo/0/protein_files/GCA_000008085.1_ASM808v1_genomic.faa:
  AE017199.1_1
  AE017199.1_2
  AE017199.1_3
  AE017199.1_4
  AE017199.1_5


## 3. Submit bakta_proteins job

One container per genome. Args: `--output /out --threads 4 --force`. No `--db` needed (the image's `BAKTA_DB` env var points to `/ref_data/db`).

In [1]:
if not input_files:
    raise RuntimeError("no .faa inputs")

job = tscli.submit_job(
    IMAGE,
    input_files,
    OUTPUT_DIR,
    cluster="kbase",
    declobber=True,
    output_mount_point="/out",
    args=[
        "--output", "/out",
        "--threads", "4",
        "--force",
        tscli.insert_files(),
    ],
    num_containers=len(input_files),
    cpus=4,
    memory="16GB",
    runtime="PT2H",
)
print(f"Job ID: {job.id}")

Job ID: 8c44fe17-bc45-4756-9847-01c5bed43f08


## 4. Wait for completion

In [1]:
t0 = time.time()
result = job.wait_for_completion()
print(f"completed in {time.time()-t0:.0f}s")
print(f"state: {job.get_job_status()['state']}")
print(f"exit codes: {job.get_exit_codes().get('exit_codes')}")

completed in 520s
state: complete
exit codes: [0, 0, 0, 0]


## 5. Inspect output files

bakta_proteins produces a per-input set of files: `.tsv`, `.gff3`, `.gbff`, `.embl`, `.faa`, `.hypotheticals.tsv`, `.hypotheticals.faa`, `.json`, `.txt`, `.log`.

In [2]:
outs = job.get_job()["outputs"]
print(f"{len(outs)} total output files")
for o in outs[:20]:
    print(f"  {o['file']}")
if len(outs) > 20:
    print(f"  ... +{len(outs)-20} more")

24 total output files
  cts/io/jplfaria/output/bakta_proteins/test/v1/0/GCA_000008085.1_ASM808v1_genomic.log
  cts/io/jplfaria/output/bakta_proteins/test/v1/0/GCA_000008085.1_ASM808v1_genomic.tsv
  cts/io/jplfaria/output/bakta_proteins/test/v1/0/GCA_000008085.1_ASM808v1_genomic.inference.tsv
  cts/io/jplfaria/output/bakta_proteins/test/v1/0/GCA_000008085.1_ASM808v1_genomic.json
  cts/io/jplfaria/output/bakta_proteins/test/v1/0/GCA_000008085.1_ASM808v1_genomic.hypotheticals.tsv
  cts/io/jplfaria/output/bakta_proteins/test/v1/0/GCA_000008085.1_ASM808v1_genomic.faa
  cts/io/jplfaria/output/bakta_proteins/test/v1/1/GCA_000010565.1_ASM1056v1_genomic.log
  cts/io/jplfaria/output/bakta_proteins/test/v1/1/GCA_000010565.1_ASM1056v1_genomic.tsv
  cts/io/jplfaria/output/bakta_proteins/test/v1/1/GCA_000010565.1_ASM1056v1_genomic.inference.tsv
  cts/io/jplfaria/output/bakta_proteins/test/v1/1/GCA_000010565.1_ASM1056v1_genomic.json
  cts/io/jplfaria/output/bakta_proteins/test/v1/1/GCA_000010565.1_AS

## 6. Read per-locus annotation TSVs as a dataframe

Same format as `cdm_bakta`'s TSV: `Sequence Id, Type, Start, Stop, Strand, Locus Tag, Gene, Product, DbXrefs`. Concatenate across the 4 genomes.

In [1]:
tsvs = [o for o in outs if o["file"].endswith(".tsv")
        and ".hypotheticals." not in o["file"]
        and ".inference." not in o["file"]
        and "/tmp/" not in o["file"]]
print(f"{len(tsvs)} bakta_proteins .tsv file(s)")

# bakta_proteins TSV format (different from bakta genome-mode):
#   # comment lines (Bakta version, DB version)
#   ID\tLength\tGene\tProduct\tEC\tGO\tCOG\tRefSeq\tUniParc\tUniRef
# pandas can skip # comment lines natively.
frames = []
for o in tsvs:
    bucket, key = o["file"].split("/", 1)
    raw = mincli.get_object(bucket, key).read().decode("utf-8")
    df = pd.read_csv(io.StringIO(raw), sep="\t", comment="#")
    df["source_file"] = o["file"]
    frames.append(df)

all_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f"\nTotal rows (= total input proteins): {len(all_df)}")
if len(all_df):
    n_real = (all_df["Product"] != "hypothetical protein").sum()
    n_gene = all_df["Gene"].notna().sum()
    n_ec   = all_df["EC"].notna().sum()
    print(f"  with real Product (non-hypothetical): {n_real} ({100*n_real/len(all_df):.1f}%)")
    print(f"  with Gene name:                       {n_gene}")
    print(f"  with EC number:                       {n_ec}")
all_df.head(10)

4 bakta_proteins .tsv file(s)

Total rows (= total input proteins): 5802
  with real Product (non-hypothetical): 3032 (52.3%)
  with Gene name:                       1002
  with EC number:                       706


              ID  Length Gene               Product   EC   GO  COG RefSeq  \
0   AE017199.1_1     293  NaN  hypothetical protein  NaN  NaN  NaN    NaN   
1   AE017199.1_2     602  NaN  hypothetical protein  NaN  NaN  NaN    NaN   
2   AE017199.1_3     173  NaN  hypothetical protein  NaN  NaN  NaN    NaN   
3   AE017199.1_4     152  NaN  hypothetical protein  NaN  NaN  NaN    NaN   
4   AE017199.1_5     306  NaN  hypothetical protein  NaN  NaN  NaN    NaN   
5   AE017199.1_6     136  NaN  hypothetical protein  NaN  NaN  NaN    NaN   
6   AE017199.1_7     177  NaN  hypothetical protein  NaN  NaN  NaN    NaN   
7   AE017199.1_8      56  NaN  hypothetical protein  NaN  NaN  NaN    NaN   
8   AE017199.1_9     413  NaN  hypothetical protein  NaN  NaN  NaN    NaN   
9  AE017199.1_10      98  NaN  hypothetical protein  NaN  NaN  NaN    NaN   

  UniParc UniRef                                        source_file  
0     NaN    NaN  cts/io/jplfaria/output/bakta_proteins/test/v1/...  
1     NaN   

## 7. Verify locus tags preserved (the whole point of Mode B)

Pull the locus tags from one input `.faa` (the `ID` column in the output corresponds to the FASTA header). They should appear verbatim in the output `.tsv`. If bakta_proteins were generating its own tags (like `cdm_bakta` does), this would fail.

In [1]:
# Pull input locus tags from one .faa
sample_key = input_files[0].split("/", 1)[1]
raw = mincli.get_object("cts", sample_key).read().decode("utf-8", errors="replace")
input_tags = set(l[1:].split()[0] for l in raw.splitlines() if l.startswith(">"))

# Compare against output ID column (filter to the same genome's source_file)
sample_basename = sample_key.rsplit("/", 1)[-1].removesuffix(".faa")
output_tags = set(all_df[all_df["source_file"].str.contains(sample_basename)]["ID"].dropna().astype(str))
overlap = input_tags & output_tags
only_in_output = output_tags - input_tags
only_in_input  = input_tags - output_tags
print(f"input file: {sample_key}")
print(f"input tags:        {len(input_tags)}")
print(f"output IDs:        {len(output_tags)}")
print(f"overlap:           {len(overlap)}")
print(f"only in output:    {len(only_in_output)}  (first 5: {sorted(only_in_output)[:5]})")
print(f"only in input:     {len(only_in_input)}   (first 5: {sorted(only_in_input)[:5]})")
print(f"\nsample 5 input tags:  {sorted(input_tags)[:5]}")
print(f"sample 5 output IDs:  {sorted(output_tags)[:5]}")

input file: io/gavin/test_output/kb_demo/0/protein_files/GCA_000008085.1_ASM808v1_genomic.faa
input tags:        583
output IDs:        583
overlap:           583
only in output:    0  (first 5: [])
only in input:     0   (first 5: [])

sample 5 input tags:  ['AE017199.1_1', 'AE017199.1_10', 'AE017199.1_100', 'AE017199.1_101', 'AE017199.1_102']
sample 5 output IDs:  ['AE017199.1_1', 'AE017199.1_10', 'AE017199.1_100', 'AE017199.1_101', 'AE017199.1_102']


## 8. End-to-end check

If all of the below are True the new image is healthy and Mode B (proteins-in, locus tags preserved) is working.

In [1]:
preserved = False
if "ID" in all_df.columns and len(all_df):
    sample_basename = sample_key.rsplit("/", 1)[-1].removesuffix(".faa")
    output_tags = set(all_df[all_df["source_file"].str.contains(sample_basename)]["ID"].dropna().astype(str))
    preserved = len(input_tags & output_tags) >= 0.95 * len(input_tags)

n_real_products = (all_df["Product"] != "hypothetical protein").sum() if "Product" in all_df.columns else 0

checks = {
    "job complete":                     job.get_job_status()["state"] == "complete",
    "per-container .tsv files present": len(tsvs) == len(input_files),
    "non-empty annotations":            len(all_df) > 0,
    "every row has a Product":          all_df["Product"].notna().all() if "Product" in all_df.columns else False,
    "some non-hypothetical products":   n_real_products > 0,
    "locus tags preserved (>=95%)":     preserved,
}
for k, v in checks.items():
    print(f"  [{'x' if v else ' '}] {k}")
if all(checks.values()):
    print(f"\nAll green. Mode B works: locus tags preserved, bakta_proteins annotates correctly.")
    print(f"({n_real_products} of {len(all_df)} proteins got real annotations; the rest are 'hypothetical protein' which is expected for archaeal genomes against bakta's bacteria-tuned DB.)")
else:
    print("\nSomething's off, inspect outputs above.")

  [x] job complete
  [x] per-container .tsv files present
  [x] non-empty annotations
  [x] every row has a Product
  [x] some non-hypothetical products
  [x] locus tags preserved (>=95%)

All green. Mode B works: locus tags preserved, bakta_proteins annotates correctly.
(3032 of 5802 proteins got real annotations; the rest are 'hypothetical protein' which is expected for archaeal genomes against bakta's bacteria-tuned DB.)
